# Agro Inteligente — Análise de Impacto Climático no Ativo SLCE3

**Empresa:** SLC Agrícola (SLCE3) · **Período:** Jan–Jun 2025 · **Região:** Sorriso, MT  
**Autores:** Lucas Lellis Barros, Gabriel Pires Pinheiro Correa, Hannah Gabrielly Marques Pompeu  
**Turma:** 2023/2 — 6º Semestre EN. Software

---

Este notebook analisa a correlação entre variáveis meteorológicas e o desempenho do ativo SLCE3 na B3.

## 1. Importação de Bibliotecas

Antes de fazer qualquer coisa em Python, precisamos importar as bibliotecas — pense nelas como ferramentas que alguém já construiu e que vamos usar.

- **openpyxl** → lê arquivos Excel (.xlsx) direto pelo Python
- **pandas** → manipula tabelas de dados (parecido com Excel, mas no código)
- **numpy** → faz cálculos matemáticos avançados (médias, regressão linear, etc.)
- **plotly** → gera gráficos interativos que abrem no navegador
- **scipy.stats** → calcula a correlação de Pearson entre duas variáveis
- **warnings** → controla mensagens de aviso — usamos para silenciar alertas que não afetam o resultado

In [ ]:
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

import openpyxl
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

## 2. Leitura dos Dados

Aqui abrimos o arquivo Excel `dados.xlsx` usando o `openpyxl` e transformamos em um DataFrame do pandas.

**O que é um DataFrame?**  
É uma tabela — tem linhas e colunas, igual a uma planilha. Cada linha é um pregão (dia de negociação) e cada coluna é uma variável (preço, temperatura, etc.).

**Por que converter para numérico?**  
O Excel às vezes armazena números como texto. Se tentarmos fazer contas com texto, dá erro. O `pd.to_numeric(..., errors='coerce')` converte o que for número e transforma em vazio (`NaN`) o que não for.

**O que é `NaN`?**  
Sigla de *Not a Number* — representa um valor ausente. Vamos tratar esses casos nas análises.

In [ ]:
wb = openpyxl.load_workbook('dados.xlsx', read_only=True)
ws = wb.active

linhas = list(ws.iter_rows(values_only=True))
df = pd.DataFrame(linhas[1:], columns=linhas[0])

df['data'] = pd.to_datetime(df['data'])
df = df.sort_values('data').reset_index(drop=True)

colunas_numericas = [
    'preco_abertura', 'preco_maximo', 'preco_minimo', 'preco_fechamento',
    'volume', 'retorno_diario', 'media_movel_7d', 'temperatura_media',
    'temperatura_maxima', 'temperatura_minima', 'precipitacao_mm',
    'velocidade_vento', 'chuva_acumulada_7d', 'precipitacao_lag_7d'
]
df[colunas_numericas] = df[colunas_numericas].apply(pd.to_numeric, errors='coerce')

df.head()

## 3. Métricas Derivadas

O arquivo original tem preços e dados climáticos, mas não tem alguns indicadores financeiros importantes. Calculamos três métricas novas:

**Drawdown**  
Mede quanto o preço caiu em relação ao maior valor que já atingiu até aquele momento. Se o preço foi a R$18 e depois caiu para R$15, o drawdown é -16,7%. Serve para responder: *qual foi a pior queda que o investidor enfrentou?*

Fórmula: `(preço atual - pico histórico) / pico histórico × 100`

**Retorno Acumulado**  
Se você tivesse comprado no primeiro dia e segurado até o fim, quanto teria ganho ou perdido? Calculamos multiplicando os retornos diários em cadeia — isso se chama produto acumulado.

**Volatilidade 7d**  
O desvio padrão dos últimos 7 retornos diários. Quando esse número é alto, o preço está oscilando muito — sinal de instabilidade. Serve para responder: *o clima aumenta o risco do ativo?*

**Agregação mensal**  
Agrupamos os dados por mês para enxergar tendências maiores — preço médio, volume total, temperatura média, precipitação etc. por mês.

In [ ]:
df['drawdown'] = ((df['preco_fechamento'] - df['preco_fechamento'].cummax()) / df['preco_fechamento'].cummax() * 100).round(4)
df['retorno_acumulado'] = ((1 + df['retorno_diario'].fillna(0)).cumprod() - 1) * 100
df['volatilidade_7d'] = df['retorno_diario'].rolling(7).std() * 100

meses_map = {1:'Jan', 2:'Fev', 3:'Mar', 4:'Abr', 5:'Mai', 6:'Jun'}
df['nome_mes'] = df['mes'].map(meses_map)

mensal = df.groupby('mes').agg(
    nome_mes=('nome_mes', 'first'),
    preco_medio=('preco_fechamento', 'mean'),
    volume_total=('volume', 'sum'),
    retorno_medio=('retorno_diario', 'mean'),
    temp_media=('temperatura_media', 'mean'),
    precip_media=('precipitacao_mm', 'mean'),
    precip_total=('precipitacao_mm', 'sum'),
    drawdown_medio=('drawdown', 'mean')
).reset_index()

## 4. KPIs — Indicadores-Chave de Performance

KPI significa *Key Performance Indicator* — são os números que resumem o desempenho do ativo e do clima no semestre. Montamos um dicionário Python com cada indicador e depois exibimos como tabela.

Alguns que merecem atenção:
- **Variação Total**: se positiva, o ativo valorizou no semestre
- **Volatilidade Diária**: quanto o retorno oscila em média por dia — quanto maior, mais arriscado
- **Drawdown Máximo**: a maior queda do pico — o pior momento para quem estava investido

In [ ]:
kpis = {
    'Período':                   f"{df['data'].min().strftime('%d/%m/%Y')} a {df['data'].max().strftime('%d/%m/%Y')}",
    'Total de Pregões':          len(df),
    'Preço Inicial (R$)':        round(df.iloc[0]['preco_fechamento'], 2),
    'Preço Final (R$)':          round(df.iloc[-1]['preco_fechamento'], 2),
    'Variação Total (%)':        round(((df.iloc[-1]['preco_fechamento'] / df.iloc[0]['preco_fechamento']) - 1) * 100, 2),
    'Volatilidade Diária (%)':   round(df['retorno_diario'].std() * 100, 2),
    'Drawdown Máximo (%)':       round(df['drawdown'].min(), 2),
    'Melhor Dia (%)':            round(df['retorno_diario'].max() * 100, 2),
    'Pior Dia (%)':              round(df['retorno_diario'].min() * 100, 2),
    'Temperatura Média (°C)':    round(df['temperatura_media'].mean(), 1),
    'Precipitação Total (mm)':   round(df['precipitacao_mm'].sum(), 1),
}

pd.DataFrame(kpis.items(), columns=['Indicador', 'Valor'])

## 5. Correlações de Pearson — Clima x Mercado

**O que é correlação de Pearson?**  
É uma medida estatística que diz se duas variáveis se movem juntas. O resultado é um número `r` entre -1 e +1:

- `r = +1` → quando uma sobe, a outra sobe exatamente junto (correlação perfeita positiva)
- `r = -1` → quando uma sobe, a outra cai exatamente junto (correlação perfeita negativa)
- `r = 0` → não há relação nenhuma entre as duas

**Classificação da força:**
- `|r| < 0,10` → Desprezível (praticamente sem relação)
- `0,10 ≤ |r| < 0,30` → Fraca
- `0,30 ≤ |r| < 0,50` → Moderada
- `|r| ≥ 0,50` → Forte

**O que é p-valor?**  
Indica se a correlação encontrada é estatisticamente significativa ou pode ter acontecido por acaso. Convencionalmente, p-valor < 0,05 indica que o resultado é confiável.

**O que é `.dropna()`?**  
Remove linhas com valores ausentes antes de calcular — necessário porque `stats.pearsonr` não aceita NaN.

In [ ]:
variaveis_clima = {
    'temperatura_media':   'Temperatura Média',
    'temperatura_maxima':  'Temperatura Máxima',
    'temperatura_minima':  'Temperatura Mínima',
    'precipitacao_mm':     'Precipitação Diária',
    'chuva_acumulada_7d':  'Chuva Acumulada 7d',
    'precipitacao_lag_7d': 'Precipitação Lag 7d',
    'velocidade_vento':    'Velocidade do Vento',
}

resultados_corr = []
for coluna, nome in variaveis_clima.items():
    sub = df[[coluna, 'preco_fechamento', 'retorno_diario']].dropna()
    r_preco,  p_preco  = stats.pearsonr(sub[coluna], sub['preco_fechamento'])
    r_retorno, p_retorno = stats.pearsonr(sub[coluna], sub['retorno_diario'])
    abs_r = abs(r_preco)
    forca = 'Desprezível' if abs_r < 0.1 else 'Fraca' if abs_r < 0.3 else 'Moderada' if abs_r < 0.5 else 'Forte'
    resultados_corr.append({
        'Variável Climática':    nome,
        'r (vs preço)':          round(r_preco, 4),
        'r (vs retorno)':        round(r_retorno, 4),
        'p-valor':               round(p_preco, 4),
        'Força':                 forca,
        'Direção':               'Positiva' if r_preco > 0 else 'Negativa'
    })

df_corr = pd.DataFrame(resultados_corr).sort_values('r (vs preço)', key=abs, ascending=False)
df_corr

## 6. Visualizações

### 6.1 Preço de Fechamento e Média Móvel 7 Dias

**Por que esse gráfico?**  
É o gráfico base de qualquer análise de ativo financeiro. O preço diário tem muito ruído — sobe e desce por motivos pontuais. A média móvel de 7 dias suaviza esse ruído e mostra a tendência real.

**Como funciona a média móvel?**  
Para cada dia, calcula a média dos últimos 7 preços de fechamento. Isso já estava calculado no arquivo — é a coluna `media_movel_7d`.

**Como ler o gráfico?**  
Quando o preço (azul) fica acima da média móvel (verde), o ativo está em alta. Quando fica abaixo, está em queda. O cruzamento das duas linhas é um sinal clássico de reversão de tendência.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df['data'], y=df['preco_fechamento'],
                         name='Preço de Fechamento',
                         line=dict(color='#3b82f6', width=2)))
fig.add_trace(go.Scatter(x=df['data'], y=df['media_movel_7d'],
                         name='Média Móvel 7d',
                         line=dict(color='#22c55e', width=1.5, dash='dash')))
fig.update_layout(title='SLCE3 — Preço de Fechamento e Média Móvel 7 Dias',
                  xaxis_title='Data', yaxis_title='Preço (R$)',
                  height=400, hovermode='x unified')
fig

### 6.2 Retorno Diário

**Por que esse gráfico?**  
Mostra os ganhos e perdas dia a dia. Verde = dia positivo, vermelho = dia negativo. Permite identificar visualmente os momentos de maior volatilidade — quando as barras são muito altas ou muito baixas.

**Como calculamos o retorno diário?**  
Já veio calculado no arquivo: `(preço hoje - preço ontem) / preço ontem`. Multiplicamos por 100 para exibir em percentual.

In [ ]:
cores = ['#22c55e' if v >= 0 else '#ef4444' for v in df['retorno_diario'].fillna(0)]
fig = go.Figure(go.Bar(x=df['data'], y=df['retorno_diario'] * 100, marker_color=cores))
fig.update_layout(title='SLCE3 — Retorno Diário (%)',
                  xaxis_title='Data', yaxis_title='Retorno (%)', height=380)
fig

### 6.3 Drawdown Acumulado

**Por que esse gráfico?**  
Responde diretamente à pergunta do projeto: *existe relação entre clima e queda abrupta?* O drawdown mostra quanto o preço caiu do seu pico histórico até aquele momento.

**Como ler o gráfico?**  
O eixo Y sempre é zero ou negativo. Quando a área vermelha vai fundo, significa que o ativo estava longe do seu topo. Quando volta a zero, o preço atingiu um novo pico.

**O que observar?**  
Se os períodos de drawdown profundo coincidem com períodos de seca ou temperatura extrema, isso sustenta a hipótese do projeto.

In [ ]:
fig = go.Figure(go.Scatter(x=df['data'], y=df['drawdown'],
                           fill='tozeroy', mode='lines',
                           line=dict(color='#ef4444', width=2),
                           fillcolor='rgba(239,68,68,0.15)',
                           name='Drawdown (%)'))
fig.update_layout(title='SLCE3 — Drawdown Acumulado (%)',
                  xaxis_title='Data', yaxis_title='Drawdown (%)', height=380)
fig

### 6.4 Volume Negociado por Mês

**Por que esse gráfico?**  
Volume alto significa mais compradores e vendedores ativos — indica interesse do mercado naquele período. Um pico de volume junto com uma queda de preço pode indicar pânico de venda. Junto com alta de preço, pode indicar euforia.

**Como ler?**  
Barra mais alta = mais negociações naquele mês. Cruzar isso com o preço médio do mês revela se o mercado estava comprando na alta ou vendendo na baixa.

In [ ]:
fig = px.bar(mensal, x='nome_mes', y='volume_total',
             title='SLCE3 — Volume Total Negociado por Mês',
             labels={'nome_mes': 'Mês', 'volume_total': 'Volume'},
             color='volume_total', color_continuous_scale='Blues')
fig.update_layout(height=380, coloraxis_showscale=False)
fig

### 6.5 Precipitação vs Preço de Fechamento

**Por que esse gráfico?**  
Responde: *como a precipitação impacta o preço?* Usamos eixo duplo (dual axis) porque preço e chuva têm escalas completamente diferentes — preço em R$ e chuva em mm. Com eixo duplo, as duas séries ficam visíveis proporcionalmente.

**O que é `secondary_y=True`?**  
Cria um segundo eixo Y do lado direito do gráfico. A linha azul (preço) usa o eixo da esquerda e as barras de chuva usam o eixo da direita.

**O que observar?**  
Se as barras de chuva sobem e o preço cai juntos, isso sustenta a correlação negativa encontrada no Pearson.

In [ ]:
fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Scatter(x=df['data'], y=df['preco_fechamento'],
                         name='Preço de Fechamento',
                         line=dict(color='#3b82f6', width=2)), secondary_y=False)
fig.add_trace(go.Bar(x=df['data'], y=df['precipitacao_mm'],
                     name='Precipitação Diária (mm)',
                     marker_color='rgba(96,165,250,0.4)'), secondary_y=True)
fig.add_trace(go.Scatter(x=df['data'], y=df['chuva_acumulada_7d'],
                         name='Chuva Acumulada 7d (mm)',
                         line=dict(color='#22c55e', width=1.5, dash='dot')), secondary_y=True)
fig.update_layout(title='Precipitação vs Preço de Fechamento — SLCE3',
                  height=420, hovermode='x unified')
fig.update_yaxes(title_text='Preço (R$)', secondary_y=False)
fig.update_yaxes(title_text='Precipitação (mm)', secondary_y=True)
fig

### 6.6 Temperatura Média vs Preço de Fechamento

**Por que esse gráfico?**  
A temperatura foi a variável com maior correlação de Pearson com o preço (r = +0,32 — moderada positiva). Esse gráfico mostra visualmente essa relação ao longo do tempo.

**O que observar?**  
Qualquer padrão visual de sincronia entre a linha laranja (temperatura) e a azul (preço) reforça a correlação encontrada. Nos meses mais frios (maio/junho em MT), veja o que acontece com o preço.

In [ ]:
fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Scatter(x=df['data'], y=df['preco_fechamento'],
                         name='Preço de Fechamento',
                         line=dict(color='#3b82f6', width=2)), secondary_y=False)
fig.add_trace(go.Scatter(x=df['data'], y=df['temperatura_media'],
                         name='Temperatura Média (°C)',
                         line=dict(color='#f97316', width=1.5)), secondary_y=True)
fig.update_layout(title='Temperatura Média vs Preço de Fechamento — SLCE3',
                  height=420, hovermode='x unified')
fig.update_yaxes(title_text='Preço (R$)', secondary_y=False)
fig.update_yaxes(title_text='Temperatura (°C)', secondary_y=True)
fig

### 6.7 Dispersão — Temperatura x Preço

**O que é um gráfico de dispersão (scatter)?**  
Cada ponto representa um dia. O eixo X é a temperatura daquele dia, o eixo Y é o preço de fechamento. Serve para ver se existe padrão entre as duas variáveis.

**O que é a linha de tendência?**  
Calculamos uma regressão linear simples com `numpy.polyfit` — ela encontra a reta que melhor descreve a relação entre temperatura e preço. Se a reta sobe da esquerda para a direita, a correlação é positiva. Se desce, é negativa.

**Por que não usamos `trendline='ols'` do plotly?**  
Essa opção exige a biblioteca `statsmodels` que não vem instalada por padrão. Fizemos o mesmo cálculo manualmente com numpy para evitar dependências extras.

In [ ]:
sub = df[['temperatura_media', 'preco_fechamento', 'mes']].dropna()
x, y = sub['temperatura_media'].values, sub['preco_fechamento'].values
m, b = np.polyfit(x, y, 1)
x_linha = np.linspace(x.min(), x.max(), 100)

fig = px.scatter(df, x='temperatura_media', y='preco_fechamento',
                 color='mes', hover_data=['data'],
                 labels={'temperatura_media': 'Temperatura Média (°C)',
                         'preco_fechamento': 'Preço de Fechamento (R$)',
                         'mes': 'Mês'},
                 title='Dispersão: Temperatura Média x Preço de Fechamento')
fig.add_trace(go.Scatter(x=x_linha, y=m*x_linha+b, mode='lines',
                         name='Tendência',
                         line=dict(color='black', width=1.5, dash='dash')))
fig.update_layout(height=420)
fig

### 6.8 Dispersão — Precipitação x Preço

**Por que esse gráfico?**  
Mesma lógica do scatter anterior, mas para precipitação. A linha de tendência descendente confirma visualmente a correlação negativa fraca (r = -0,23) — dias com mais chuva tendem a ter preços ligeiramente menores.

**O que são os pontos extremos no eixo X?**  
Dias com chuva muito intensa (acima de 30mm). Note que nesses dias o preço não segue um padrão claro — isso é parte do motivo pelo qual a correlação é fraca, não forte.

In [ ]:
sub = df[['precipitacao_mm', 'preco_fechamento']].dropna()
x, y = sub['precipitacao_mm'].values, sub['preco_fechamento'].values
m, b = np.polyfit(x, y, 1)
x_linha = np.linspace(x.min(), x.max(), 100)

fig = px.scatter(df, x='precipitacao_mm', y='preco_fechamento',
                 color='mes', hover_data=['data'],
                 labels={'precipitacao_mm': 'Precipitação Diária (mm)',
                         'preco_fechamento': 'Preço de Fechamento (R$)',
                         'mes': 'Mês'},
                 title='Dispersão: Precipitação Diária x Preço de Fechamento')
fig.add_trace(go.Scatter(x=x_linha, y=m*x_linha+b, mode='lines',
                         name='Tendência',
                         line=dict(color='black', width=1.5, dash='dash')))
fig.update_layout(height=420)
fig

### 6.9 Correlações de Pearson — Gráfico de Barras

**Por que esse gráfico?**  
Responde diretamente a pergunta central do projeto: *qual variável climática tem maior correlação com o preço?* Em vez de uma tabela com números, as barras horizontais tornam a comparação imediata e visual.

**Como ler?**  
- Barra verde e à direita = correlação positiva (quando a variável sobe, o preço tende a subir)
- Barra vermelha e à esquerda = correlação negativa (quando a variável sobe, o preço tende a cair)
- Barra mais longa = correlação mais forte

In [ ]:
df_corr_ord = df_corr.sort_values('r (vs preço)')
cores_barras = ['#ef4444' if v < 0 else '#22c55e' for v in df_corr_ord['r (vs preço)']]

fig = go.Figure(go.Bar(
    x=df_corr_ord['r (vs preço)'],
    y=df_corr_ord['Variável Climática'],
    orientation='h',
    marker_color=cores_barras,
    text=df_corr_ord['r (vs preço)'].apply(lambda x: f'{x:+.4f}'),
    textposition='outside'
))
fig.update_layout(
    title='Correlação de Pearson — Variáveis Climáticas vs Preço de Fechamento SLCE3',
    xaxis_title='Coeficiente de Pearson (r)',
    height=400, xaxis=dict(range=[-0.5, 0.5])
)
fig

### 6.10 Volatilidade 7d vs Chuva Acumulada

**Por que esse gráfico?**  
Responde: *o clima aumenta o risco do ativo?* Comparamos a volatilidade móvel de 7 dias (risco) com a chuva acumulada no mesmo período. Se a volatilidade sobe junto com os picos de chuva, o clima está influenciando a instabilidade do preço.

**O que é `rolling(7).std()`?**  
Para cada dia, calcula o desvio padrão dos 7 retornos anteriores. Desvio padrão mede o quanto os valores variam em torno da média — quanto maior, mais o preço oscilou.

In [ ]:
fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Scatter(x=df['data'], y=df['volatilidade_7d'],
                         name='Volatilidade 7d (%)',
                         line=dict(color='#f59e0b', width=2)), secondary_y=False)
fig.add_trace(go.Scatter(x=df['data'], y=df['chuva_acumulada_7d'],
                         name='Chuva Acumulada 7d (mm)',
                         line=dict(color='#60a5fa', width=1.5, dash='dot')), secondary_y=True)
fig.update_layout(title='Volatilidade 7d vs Chuva Acumulada — Relação Clima x Risco',
                  height=400, hovermode='x unified')
fig.update_yaxes(title_text='Volatilidade (%)', secondary_y=False)
fig.update_yaxes(title_text='Chuva Acumulada (mm)', secondary_y=True)
fig

### 6.11 Resumo Mensal — Preço e Precipitação

**Por que esse gráfico?**  
Responde: *períodos de plantio/colheita impactam o preço?* Janeiro e fevereiro são colheita de soja em Mato Grosso — período de alta oferta, preços menores. Abril, já no período seco pré-inverno, registrou o pico do semestre.

**O que é `make_subplots(rows=1, cols=2)`?**  
Cria dois gráficos lado a lado numa mesma figura. `row=1, col=1` coloca o gráfico na primeira posição, `row=1, col=2` na segunda.

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Preço Médio de Fechamento por Mês (R$)',
                                    'Precipitação Total por Mês (mm)'))
fig.add_trace(go.Bar(x=mensal['nome_mes'], y=mensal['preco_medio'].round(2),
                     marker_color='#3b82f6',
                     text=mensal['preco_medio'].round(2),
                     textposition='outside'), row=1, col=1)
fig.add_trace(go.Bar(x=mensal['nome_mes'], y=mensal['precip_total'].round(1),
                     marker_color='#60a5fa',
                     text=mensal['precip_total'].round(1),
                     textposition='outside'), row=1, col=2)
fig.update_layout(height=400, showlegend=False,
                  title='Resumo Mensal — SLCE3 | Jan–Jun 2025')
fig

## 7. Conclusões

A célula abaixo gera as respostas às perguntas de negócio automaticamente a partir dos dados calculados — não são textos fixos, são calculados na hora com os valores reais.

**Como funciona?**  
Buscamos os valores já calculados nas variáveis `df_corr`, `mensal` e `kpis` e montamos um dicionário com as respostas. Depois exibimos como tabela.

In [ ]:
topo       = df_corr.iloc[0]
r_chuva    = df_corr[df_corr['Variável Climática'] == 'Precipitação Diária']['r (vs preço)'].values[0]
r_lag      = df_corr[df_corr['Variável Climática'] == 'Precipitação Lag 7d']['r (vs preço)'].values[0]
dd_junho   = mensal[mensal['mes'] == 6]['drawdown_medio'].values[0]

conclusoes = {
    'Variável com maior correlação':  f"{topo['Variável Climática']} (r = {topo['r (vs preço)']}) — {topo['Força']} {topo['Direção']}",
    'Impacto da chuva no preço':      f"r = {r_chuva} (Fraca Negativa) — chuvas intensas coincidem com leve queda no preço",
    'Efeito defasado (lag 7d)':       f"r = {r_lag} (Desprezível) — o impacto climático é imediato, não defasado",
    'Seca prolongada e drawdown':     f"Mai–Jun com precipitação quase zero; drawdown médio em Junho = {dd_junho:.1f}% (pior do semestre)",
    'Variação total do ativo':        f"{kpis['Variação Total (%)']:+.2f}%",
    'Drawdown máximo':                f"{kpis['Drawdown Máximo (%)']}%",
    'Volatilidade diária':            f"{kpis['Volatilidade Diária (%)']}%",
}

pd.DataFrame(conclusoes.items(), columns=['Pergunta de Negócio', 'Resposta'])